In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, roc_curve, precision_score, recall_score, f1_score
from lightgbm import LGBMClassifier
import joblib
import pickle
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

In [ ]:
# Load optimized data
with open('../../data/train_optimized.pkl', 'rb') as f:
    df = pickle.load(f)

print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## Feature Selection & Grouping

In [ ]:
# Kategorize columns
categorical_cols = [col for col in df.columns if df[col].dtype in ['object', 'category']]
categorical_but_high_cardinality = [col for col in categorical_cols if df[col].nunique() > 50]

numeric_cols = [col for col in df.columns if df[col].dtype in ['int64', 'float64','Int32','Int8','Int16']]
numeric_but_categorical = [col for col in df.columns if df[col].dtype in ['int64', 'float64','Int32','Int8','Int16'] and df[col].nunique() < 20 and col not in ['isFraud', 'TransactionID']]

print(f"Categorical columns: {len(categorical_cols)}")
print(f"High cardinality categoricals (>50): {len(categorical_but_high_cardinality)}")
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Numeric but categorical (<20): {len(numeric_but_categorical)}")

In [ ]:
# Prepare target and features
df = df.set_index('TransactionID')
y = df['isFraud']
df = df.drop('isFraud', axis=1)

## Feature Engineering

In [ ]:
# 1. Label Encoding for low cardinality
categorical_cols = [col for col in categorical_cols if col not in categorical_but_high_cardinality]
numeric_only = [col for col in numeric_cols if col not in numeric_but_categorical]
high_card_cols = categorical_but_high_cardinality

le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

# 2. Numeric-as-categorical encoding
for col in numeric_but_categorical:
    if df[col].nunique() < 10:
        df = pd.get_dummies(df, columns=[col], prefix=col, drop_first=True)
    else:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        le_dict[col] = le

# 3. Frequency encoding for high cardinality
freq_maps = {}
for col in high_card_cols:
    freq_map = df[col].value_counts().to_dict()
    df[f'{col}_freq'] = df[col].map(freq_map)
    df = df.drop(col, axis=1)
    freq_maps[col] = freq_map

## Model Training

In [ ]:
# Train/validation split
time_col = 'TransactionDT'
X = df.drop([time_col], axis=1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {len(X_train)}")
print(f"Val size: {len(X_val)}")

In [ ]:
# LightGBM model
lgb_model = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=7, num_leaves=31, random_state=42, class_weight='balanced', verbose=-1)

lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='auc')
print("Training complete")

## Model Evaluation

In [ ]:
# Predictions
y_train_pred_proba = lgb_model.predict_proba(X_train)[:, 1]
y_val_pred_proba = lgb_model.predict_proba(X_val)[:, 1]
y_val_pred = lgb_model.predict(X_val)

# Metrics
train_auc = roc_auc_score(y_train, y_train_pred_proba)
val_auc = roc_auc_score(y_val, y_val_pred_proba)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

print(f"Train ROC-AUC: {train_auc:.4f}")
print(f"Val ROC-AUC: {val_auc:.4f}")
print(f"Val Precision: {precision:.4f}")
print(f"Val Recall: {recall:.4f}")
print(f"Val F1-Score: {f1:.4f}")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({'feature': X_train.columns, 'importance': lgb_model.feature_importances_}).sort_values('importance', ascending=False)

print("\nTop 20 Important Features:")
print(feature_importance.head(20))

## Save Model

In [ ]:
# Save artifacts
import os
os.makedirs('../../models', exist_ok=True)

artifacts = {
    'model': lgb_model,
    'le_dict': le_dict,
    'freq_maps': freq_maps,
    'features': X_train.columns.tolist(),
    'val_auc': val_auc,
    'metrics': {'train_auc': train_auc, 'val_auc': val_auc, 'precision': precision, 'recall': recall, 'f1': f1}
}

joblib.dump(artifacts, '../../models/baseline_lightgbm_v1.pkl')
print("✓ Model saved: models/baseline_lightgbm_v1.pkl")